In [ ]:
# 1. Install Androguard
!pip install androguard -q

# 2. Download 3 legal, open-source APKs from F-Droid for testing
!wget -q https://f-droid.org/repo/org.fdroid.fdroid_1019050.apk -O fdroid.apk
!wget -q https://f-droid.org/repo/com.termux_118.apk -O termux.apk
!wget -q https://f-droid.org/repo/org.wikipedia_50410.apk -O wikipedia.apk

print("Setup complete. APKs downloaded.")

Setup complete. APKs downloaded.


In [ ]:
import os
import csv
from androguard.core.apk import APK
from androguard.core.analysis.analysis import Analysis

# Define a list of dangerous permissions and their expected API classes
DANGEROUS_PERMISSIONS = {
    "android.permission.READ_SMS": "Landroid/telephony/SmsManager;",
    "android.permission.SEND_SMS": "Landroid/telephony/SmsManager;",
    "android.permission.ACCESS_FINE_LOCATION": "Landroid/location/LocationManager;",
    "android.permission.CAMERA": "Landroid/hardware/Camera;",
    "android.permission.RECORD_AUDIO": "Landroid/media/MediaRecorder;",
    "android.permission.READ_CONTACTS": "Landroid/provider/ContactsContract;"
}

def analyze_apk(apk_path):
    try:
        apk = APK(apk_path)
        declared_perms = apk.get_permissions()

        # Get DEX analysis to check for API usage
        dex = apk.get_dex()
        dx = Analysis()
        dx.add(dex)
        dx.create_xref()

        # Get all classes referenced in the app
        referenced_classes = [str(cls.get_name()) for cls in dx.get_classes()]

        flagged = []
        for perm, api_class in DANGEROUS_PERMISSIONS.items():
            if perm in declared_perms:
                # Check if the expected API class is actually used
                if api_class not in referenced_classes:
                    flagged.append(perm)

        return declared_perms, flagged
    except Exception as e:
        return [f"ERROR: {str(e)}"], []

# Process the downloaded APKs
apk_files = ["fdroid.apk", "termux.apk", "wikipedia.apk"]
results = []

print(f"{'APK':<15} | {'Flagged (Over-requested)':<50}")
print("-" * 70)

for apk_file in apk_files:
    if os.path.exists(apk_file):
        declared, flagged = analyze_apk(apk_file)
        if flagged:
            flag_str = ", ".join(flagged)
            print(f"{apk_file:<15} | {flag_str}")
            results.append([apk_file, ";".join(declared), ";".join(flagged)])
        else:
            print(f"{apk_file:<15} | None found")
            results.append([apk_file, ";".join(declared), "None"])

# Save results to CSV
with open("permission_analysis_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["APK", "All_Declared_Permissions", "Flagged_Over_Requests"])
    writer.writerows(results)

print("\nResults saved to permission_analysis_results.csv")

APK             | Flagged (Over-requested)                          
----------------------------------------------------------------------


Streaming output truncated to the last 5000 lines.
2026-11-24 05:57:49.734 +00:00 | DEBUG    | androguard.core.axml:getAttributeName:997 - 1
2026-11-24 05:57:49.736 +00:00 | DEBUG    | androguard.core.axml:getAttributeValueType:1027 - 1
2026-11-24 05:57:49.737 +00:00 | DEBUG    | androguard.core.axml:getAttributeValueData:1038 - 1
2026-11-24 05:57:49.737 +00:00 | DEBUG    | androguard.core.axml:getAttributeValue:1052 - 1
2026-11-24 05:57:49.740 +00:00 | DEBUG    | androguard.core.axml:__init__:1195 - found an attribute: {http://schemas.android.com/apk/res/android}name='b'org.fdroid.fdroid.nearby.SwapWorkflowActivity''
2026-11-24 05:57:49.742 +00:00 | DEBUG    | androguard.core.axml:getAttributeNamespace:981 - 2
2026-11-24 05:57:49.743 +00:00 | DEBUG    | androguard.core.axml:getAttributeUri:968 - 2
2026-11-24 05:57:49.745 +00:00 | DEBUG    | androguard.core.axml:getAttributeName:997 - 2
2026-11-24 05:57:49.746 +00:00 | DEBUG    | androguard.core.axml:getAttributeValueType:1027 - 2
2026

fdroid.apk      | None found


2026-11-24 05:57:55.421 +00:00 | INFO     | androguard.core.apk:_apk_analysis:420 - Starting analysis on AndroidManifest.xml
2026-11-24 05:57:55.422 +00:00 | DEBUG    | androguard.core.axml:__init__:1136 - AXMLPrinter
2026-11-24 05:57:55.423 +00:00 | DEBUG    | androguard.core.axml:__init__:449 - AXMLParser
2026-11-24 05:57:55.424 +00:00 | DEBUG    | androguard.core.axml:__init__:482 - FIRST HEADER <ARSCHeader idx='0x00000000' type='3' header_size='8' size='12768'>
2026-11-24 05:57:55.424 +00:00 | DEBUG    | androguard.core.axml:__init__:540 - STRING_POOL <ARSCHeader idx='0x00000008' type='1' header_size='28' size='5892'>
2026-11-24 05:57:55.426 +00:00 | DEBUG    | androguard.core.axml:is_valid:575 - True
2026-11-24 05:57:55.426 +00:00 | DEBUG    | androguard.core.axml:_do_next:593 - M_EVENT -1
2026-11-24 05:57:55.427 +00:00 | DEBUG    | androguard.core.axml:_do_next:608 - NEXT HEADER <ARSCHeader idx='0x0000170c' type='384' header_size='8' size='160'>
2026-11-24 05:57:55.427 +00:00 | D

termux.apk      | None found
wikipedia.apk   | None found

Results saved to permission_analysis_results.csv
